# Error Analysis (OOF) Notebook

This notebook analyzes **out-of-fold (OOF)** predictions produced by your training pipeline and highlights where the model **systematically over/under-predicts**.

**How to use**
1. Train a model (so you have OOF outputs under `reports/`).
2. Open this notebook from the repo root (or adjust paths in the first cell).
3. Pick a model from the dropdown and explore residual patterns.

> If you don't have OOF files yet, run:  
> `python -m src.train --model ridge` (or other model), then come back.


In [ ]:
# =========================
# 0) Setup & paths
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

# Interactive plotting
import plotly.express as px
import plotly.graph_objects as go

REPO_ROOT = Path('.')
DATA_DIR = REPO_ROOT / 'data'
REPORTS_DIR = REPO_ROOT / 'reports'

TRAIN_PATH = DATA_DIR / 'train.csv'

assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}. Put train.csv under data/"
assert REPORTS_DIR.exists(), f"Missing {REPORTS_DIR}. Run training to generate OOF artifacts."

train = pd.read_csv(TRAIN_PATH)
print('train:', train.shape)
train.head()

In [ ]:
# =========================
# 1) Discover OOF artifacts
# =========================
def discover_oof_files(reports_dir: Path):
    patterns = [
        '*_oof.csv', '*_oof.parquet',
        '*_oof.npy', '*_oof_pred.npy', '*_oof_preds.npy',
    ]
    files = []
    for pat in patterns:
        files += list(reports_dir.glob(pat))
    # de-duplicate
    uniq = {f.resolve(): f for f in files}
    return sorted(uniq.values(), key=lambda x: x.name)

oof_files = discover_oof_files(REPORTS_DIR)
print('Found OOF artifacts:')
for f in oof_files:
    print(' -', f.name)

# Infer model names from filenames like "<model>_oof.*"
def infer_model_name(path: Path) -> str:
    name = path.stem
    # handle double extensions like .parquet? stem already strips one
    # normalize common suffixes
    for suf in ['_oof_pred', '_oof_preds', '_oof']:
        if name.endswith(suf):
            return name[: -len(suf)]
    return name.replace('_oof', '')

model_to_file = {infer_model_name(f): f for f in oof_files}
print('\nModels detected:', sorted(model_to_file.keys()))

In [ ]:
# =========================
# 2) Load OOF into a standard DataFrame
# =========================
from typing import Tuple, Optional

def _maybe_read_parquet(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)

def _maybe_read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

def _maybe_read_npy(path: Path) -> np.ndarray:
    return np.load(path)

def detect_scale(y_true: np.ndarray, y_pred: np.ndarray) -> str:
    """Heuristic: decide whether predictions are in log1p space or raw space."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # Typical SalePrice is ~50k-500k. log1p is ~10-13.
    if np.nanmedian(y_true) > 1000 and np.nanmedian(y_pred) < 50:
        return 'pred_log_true_raw'
    if np.nanmedian(y_true) < 50 and np.nanmedian(y_pred) < 50:
        return 'both_log'
    if np.nanmedian(y_true) > 1000 and np.nanmedian(y_pred) > 1000:
        return 'both_raw'
    # fallback
    return 'unknown'

def load_oof_as_df(model_name: str, path: Path, train_df: pd.DataFrame) -> pd.DataFrame:
    """Return df with columns: Id, y_true_raw, y_pred_raw, y_true_log, y_pred_log."""
    # ground truth
    if 'SalePrice' not in train_df.columns:
        raise ValueError('train.csv must contain SalePrice')
    if 'Id' not in train_df.columns:
        # If no Id, create row index Id
        base = train_df.copy()
        base['Id'] = np.arange(len(base))
    else:
        base = train_df[['Id', 'SalePrice']].copy()

    y_true_raw = base['SalePrice'].to_numpy(dtype=float)
    y_true_log = np.log1p(y_true_raw)

    # load predictions
    if path.suffix.lower() == '.parquet':
        oof = _maybe_read_parquet(path)
        # try common column names
        for col in ['oof', 'oof_pred', 'oof_preds', 'pred', 'prediction', 'y_pred']:
            if col in oof.columns:
                y_pred = oof[col].to_numpy(dtype=float)
                break
        else:
            # if single column
            if oof.shape[1] == 1:
                y_pred = oof.iloc[:, 0].to_numpy(dtype=float)
            else:
                raise ValueError(f'Cannot infer prediction column in {path.name}: {list(oof.columns)}')
    elif path.suffix.lower() == '.csv':
        oof = _maybe_read_csv(path)
        # If file already includes Id, align by Id
        if 'Id' in oof.columns and any(c in oof.columns for c in ['oof', 'pred', 'prediction', 'y_pred', 'oof_pred', 'oof_preds']):
            pred_col = next((c for c in ['oof', 'oof_pred', 'oof_preds', 'pred', 'prediction', 'y_pred'] if c in oof.columns), None)
            oof = oof[['Id', pred_col]].rename(columns={pred_col: 'y_pred'})
            merged = base.merge(oof, on='Id', how='inner')
            y_pred = merged['y_pred'].to_numpy(dtype=float)
            y_true_raw_m = merged['SalePrice'].to_numpy(dtype=float)
            y_true_log_m = np.log1p(y_true_raw_m)
            base_m = merged[['Id']].copy()
            base_m['SalePrice'] = y_true_raw_m
            base = base_m
            y_true_raw = y_true_raw_m
            y_true_log = y_true_log_m
        else:
            # try common column names or single column
            pred_col = next((c for c in ['oof', 'oof_pred', 'oof_preds', 'pred', 'prediction', 'y_pred'] if c in oof.columns), None)
            if pred_col is not None:
                y_pred = oof[pred_col].to_numpy(dtype=float)
            elif oof.shape[1] == 1:
                y_pred = oof.iloc[:, 0].to_numpy(dtype=float)
            else:
                raise ValueError(f'Cannot infer prediction column in {path.name}: {list(oof.columns)}')
    elif path.suffix.lower() == '.npy':
        y_pred = _maybe_read_npy(path).astype(float).ravel()
    else:
        raise ValueError(f'Unsupported OOF file: {path}')

    # length check
    if len(y_pred) != len(base):
        raise ValueError(f'Length mismatch: pred={len(y_pred)} vs train={len(base)} for model {model_name}. '
                         f'If your OOF file includes Id, prefer exporting it and we will align by Id.')

    scale = detect_scale(y_true_raw, y_pred)

    if scale in ('both_log',):
        y_pred_log = y_pred
        y_pred_raw = np.expm1(y_pred_log)
    elif scale in ('pred_log_true_raw',):
        y_pred_log = y_pred
        y_pred_raw = np.expm1(y_pred_log)
    elif scale in ('both_raw',):
        y_pred_raw = y_pred
        y_pred_log = np.log1p(np.maximum(y_pred_raw, 0))
    else:
        # guess based on magnitude
        if np.nanmedian(y_pred) < 50:
            y_pred_log = y_pred
            y_pred_raw = np.expm1(y_pred_log)
        else:
            y_pred_raw = y_pred
            y_pred_log = np.log1p(np.maximum(y_pred_raw, 0))

    df = pd.DataFrame({
        'Id': base['Id'].to_numpy(),
        'y_true_raw': y_true_raw,
        'y_pred_raw': y_pred_raw,
        'y_true_log': y_true_log,
        'y_pred_log': y_pred_log,
    })
    df['residual_raw'] = df['y_true_raw'] - df['y_pred_raw']
    df['residual_log'] = df['y_true_log'] - df['y_pred_log']
    df['abs_error_raw'] = np.abs(df['residual_raw'])
    df['abs_error_log'] = np.abs(df['residual_log'])
    df['ape_raw'] = df['abs_error_raw'] / np.maximum(df['y_true_raw'], 1.0)
    df['model'] = model_name
    return df

# Load all detected models into a dict
oof_dfs = {}
for m, f in model_to_file.items():
    try:
        oof_dfs[m] = load_oof_as_df(m, f, train)
        print(f"[OK] {m}: {f.name} -> {oof_dfs[m].shape}")
    except Exception as e:
        print(f"[SKIP] {m}: {f.name} -> {e}")

list(oof_dfs.keys())

In [ ]:
# =========================
# 3) Choose a model to analyze
# =========================
# If you want a dropdown widget, uncomment the next cell and install ipywidgets:
# pip install ipywidgets
# In many environments, a simple variable switch is enough.

MODEL = sorted(oof_dfs.keys())[0] if oof_dfs else None
print("Selected MODEL =", MODEL)

df = oof_dfs[MODEL].copy()
df.describe().T.head(12)

## Core Views (Interactive)

Below are the most useful interactive views for diagnosing bias:
- **Residual vs Prediction**: systematic over/under prediction, heteroskedasticity
- **Error by Price Segment**: where the model struggles (low vs high prices)
- **Top error cases**: inspect outliers for data quality or feature gaps


In [ ]:
# =========================
# 4) Residual vs Prediction (raw scale)
# =========================
fig = px.scatter(
    df,
    x="y_pred_raw",
    y="residual_raw",
    hover_data=["Id", "y_true_raw", "y_pred_raw", "ape_raw"],
    title=f"Residual vs Prediction (OOF) — {MODEL}",
)
fig.add_hline(y=0)
fig.show()

In [ ]:
# =========================
# 5) Error distribution (raw & log)
# =========================
fig1 = px.histogram(df, x="residual_raw", nbins=60, title=f"Residual Distribution (raw) — {MODEL}")
fig1.show()

fig2 = px.histogram(df, x="residual_log", nbins=60, title=f"Residual Distribution (log1p) — {MODEL}")
fig2.show()

In [ ]:
# =========================
# 6) Error by Price Segment (quantiles)
# =========================
q = pd.qcut(df["y_true_raw"], q=10, duplicates="drop")
seg = df.groupby(q, observed=True).agg(
    n=("Id", "count"),
    mae=("abs_error_raw", "mean"),
    mape=("ape_raw", "mean"),
    median_true=("y_true_raw", "median"),
).reset_index()

# Make segment label readable
seg["segment"] = seg[q.name].astype(str)

fig = px.bar(
    seg,
    x="segment",
    y="mae",
    hover_data=["n", "mape", "median_true"],
    title=f"MAE by Price Segment (OOF) — {MODEL}",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

fig = px.bar(
    seg,
    x="segment",
    y="mape",
    hover_data=["n", "mae", "median_true"],
    title=f"MAPE by Price Segment (OOF) — {MODEL}",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# =========================
# 7) Top error cases (inspect)
# =========================
top = df.sort_values("abs_error_raw", ascending=False).head(25)
top[["Id", "y_true_raw", "y_pred_raw", "abs_error_raw", "ape_raw"]].reset_index(drop=True)

In [ ]:
# =========================
# 8) Attach original features for deeper slice-and-dice (optional)
# =========================
# Join back to training features (without SalePrice) so you can segment by Neighborhood, etc.
features = train.drop(columns=["SalePrice"], errors="ignore").copy()
if "Id" not in features.columns:
    features["Id"] = np.arange(len(features))

df_full = df.merge(features, on="Id", how="left")
print(df_full.shape)
df_full.head()

In [ ]:
# =========================
# 9) Segment error by a categorical feature (Neighborhood example)
# =========================
cat_col = "Neighborhood" if "Neighborhood" in df_full.columns else None
cat_col

In [ ]:
if cat_col is not None:
    g = df_full.groupby(cat_col, observed=True).agg(
        n=("Id", "count"),
        mae=("abs_error_raw", "mean"),
        mape=("ape_raw", "mean"),
        median_price=("y_true_raw", "median"),
    ).reset_index().sort_values("mae", ascending=False)

    g_top = g.head(25)
    fig = px.bar(
        g_top,
        x=cat_col,
        y="mae",
        hover_data=["n", "mape", "median_price"],
        title=f"Top 25 {cat_col} by MAE (OOF) — {MODEL}",
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

    g_top[[cat_col, "n", "mae", "mape", "median_price"]].reset_index(drop=True)
else:
    print("Neighborhood not found. Pick another categorical column in df_full.columns.")

## Compare models (optional)

If multiple models exist in `reports/`, you can compare their OOF error distributions and segment metrics.


In [ ]:
# =========================
# 10) Compare models by overall OOF metrics
# =========================
def rmse(x): 
    x = np.asarray(x, dtype=float)
    return np.sqrt(np.mean(x**2))

summary = []
for m, d in oof_dfs.items():
    summary.append({
        "model": m,
        "RMSE_raw": rmse(d["residual_raw"]),
        "MAE_raw": float(d["abs_error_raw"].mean()),
        "MAPE_raw": float(d["ape_raw"].mean()),
        "RMSE_log": rmse(d["residual_log"]),
        "MAE_log": float(d["abs_error_log"].mean()),
    })

summary_df = pd.DataFrame(summary).sort_values("RMSE_log")
summary_df

In [ ]:
# Interactive compare: RMSE_log vs MAE_log
fig = px.scatter(
    summary_df,
    x="RMSE_log",
    y="MAE_log",
    text="model",
    hover_data=["RMSE_raw", "MAE_raw", "MAPE_raw"],
    title="Model Comparison (OOF): log-space metrics",
)
fig.show()